# AIG230 NLP (Week 3 Lab) — Notebook 2: Statistical Language Models (Train, Test, Evaluate)

This notebook focuses on **n-gram Statistical Language Models (SLMs)**:
- Train **unigram**, **bigram**, **trigram** models
- Handle **OOV** with `<UNK>`
- Apply **smoothing** (Add-k)
- Evaluate with **cross-entropy** and **perplexity**
- Do **next-word prediction** and simple **text generation**

> Industry framing: even if modern systems use neural LMs, n-gram LMs are still useful for
baselines, constrained domains, and for understanding evaluation.


## 0) Setup


In [30]:

import re
import math
import random
from collections import Counter, defaultdict
from typing import List, Tuple, Dict


## 1) Data: domain text you might see in real systems


We use short texts that resemble:
- release notes
- incident summaries
- operational runbooks
- customer support messaging

In practice, you would load thousands to millions of lines.


In [31]:

corpus = [
    "vpn disconnects frequently after windows update",
    "password reset link expired user cannot login",
    "api requests timeout when latency spikes",
    "portal returns 500 error after deployment",
    "email delivery delayed messages queued",
    "mfa prompt never arrives user stuck at login",
    "wifi drops in meeting rooms access point reboot helps",
    "outlook search not returning results index corrupted",
    "printer driver install fails with error 1603",
    "teams calls choppy audio jitter high",
    "permission denied accessing shared drive though in correct group",
    "battery drains fast after bios update power settings unchanged",
    "push notifications not working on android app",
    "mailbox full cannot receive emails auto archive not running",
]

# Train/test split at sentence level
random.seed(42)
random.shuffle(corpus)
split = int(0.75 * len(corpus))
train_texts = corpus[:split]
test_texts = corpus[split:]

len(train_texts), len(test_texts), train_texts[:2], test_texts[:2]


(10,
 4,
 ['printer driver install fails with error 1603',
  'push notifications not working on android app'],
 ['email delivery delayed messages queued',
  'vpn disconnects frequently after windows update'])

## 2) Tokenization + special tokens


We will:
- lowercase
- keep alphanumerics
- split on whitespace
- add sentence boundary tokens: `<s>` and `</s>`

We will also map rare tokens to `<UNK>` based on training frequency.


In [32]:

def tokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def add_boundaries(tokens: List[str], n: int) -> List[str]:
    # For n-grams, prepend (n-1) start tokens for simpler context handling
    return ["<s>"]*(n-1) + tokens + ["</s>"]

# Example
tokens = tokenize("Printer driver install fails with error 1603")
add_boundaries(tokens, n=3)


['<s>',
 '<s>',
 'printer',
 'driver',
 'install',
 'fails',
 'with',
 'error',
 '1603',
 '</s>']

## 3) Build vocabulary and handle OOV with <UNK>


In [33]:

# Build vocab from training data
train_tokens_flat = []
for t in train_texts:
    train_tokens_flat.extend(tokenize(t))

freq = Counter(train_tokens_flat)

# Typical practical rule: map tokens with frequency <= 1 to <UNK> in small corpora
min_count = 2
vocab = {w for w, c in freq.items() if c >= min_count}
vocab |= {"<UNK>", "<s>", "</s>"}

def replace_oov(tokens: List[str], vocab: set) -> List[str]:
    return [tok if tok in vocab else "<UNK>" for tok in tokens]

# Show OOV effect
sample = tokenize(test_texts[0])
sample, replace_oov(sample, vocab)


(['email', 'delivery', 'delayed', 'messages', 'queued'],
 ['<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>'])

## 4) Train n-gram counts (unigram, bigram, trigram)


We will compute:
- `ngram_counts[(w1,...,wn)]`
- `context_counts[(w1,...,w_{n-1})]`

Then probability:
\ndefault:  P(w_n | context) = count(context + w_n) / count(context)

This fails when an n-gram is unseen, so we add smoothing.


In [34]:

def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def train_ngram_counts(texts: List[str], n: int, vocab: set) -> Tuple[Counter, Counter]:
    ngram_counts = Counter()
    context_counts = Counter()
    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        for ng in get_ngrams(toks, n):
            ngram_counts[ng] += 1
            context = ng[:-1]
            context_counts[context] += 1
    return ngram_counts, context_counts

uni_counts, uni_ctx = train_ngram_counts(train_texts, n=1, vocab=vocab)
bi_counts, bi_ctx   = train_ngram_counts(train_texts, n=2, vocab=vocab)
tri_counts, tri_ctx = train_ngram_counts(train_texts, n=3, vocab=vocab)

len(uni_counts), len(bi_counts), len(tri_counts)


(5, 10, 17)

## 5) Add-k smoothing and probability function


Add-k smoothing (a common baseline):
\na) Add *k* to every possible next word count  
b) Normalize by context_count + k * |V|

P_k(w|h) = (count(h,w) + k) / (count(h) + k*|V|)

Where V is the vocabulary.


In [35]:

def prob_addk(ngram: Tuple[str, ...], ngram_counts: Counter, context_counts: Counter, V: int, k: float = 0.5) -> float:
    context = ngram[:-1]
    return (ngram_counts[ngram] + k) / (context_counts[context] + k * V)

V = len(vocab)
# Example: P("login" | "<s>") in bigram model
example = ("<s>", "login")
prob_addk(example, bi_counts, bi_ctx, V, k=0.5)


0.038461538461538464

## 6) Evaluate: cross-entropy and perplexity on test set


We evaluate an LM by how well it predicts held-out text.

Cross-entropy (average negative log probability):
H = - (1/N) * sum log2 P(w_i | context)

Perplexity:
PP = 2^H

Lower perplexity is better.


In [36]:

def evaluate_perplexity(texts: List[str], n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, k: float = 0.5) -> float:
    V = len(vocab)
    log2_probs = []
    token_count = 0

    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        ngrams = get_ngrams(toks, n)
        for ng in ngrams:
            p = prob_addk(ng, ngram_counts, context_counts, V, k=k)
            log2_probs.append(math.log(p, 2))
            token_count += 1

    H = -sum(log2_probs) / token_count
    PP = 2 ** H
    return PP

pp_uni = evaluate_perplexity(test_texts, n=1, ngram_counts=uni_counts, context_counts=uni_ctx, vocab=vocab, k=0.5)
pp_bi  = evaluate_perplexity(test_texts, n=2, ngram_counts=bi_counts,  context_counts=bi_ctx,  vocab=vocab, k=0.5)
pp_tri = evaluate_perplexity(test_texts, n=3, ngram_counts=tri_counts, context_counts=tri_ctx, vocab=vocab, k=0.5)

pp_uni, pp_bi, pp_tri


(1.8224739937573897, 1.8712095221558311, 1.9552746520172757)

## 7) Next-word prediction (top-k)


Given a context, compute the probability of each candidate next token and return the top-k.

This mirrors:
- autocomplete in constrained domains
- template suggestion systems
- command prediction in runbooks


In [37]:

def next_word_topk(context_tokens: List[str], n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, k_smooth: float = 0.5, top_k: int = 5):
    # Context length should be n-1
    V = len(vocab)
    context = tuple(context_tokens[-(n-1):]) if n > 1 else tuple()
    candidates = []
    for w in vocab:
        if w in {"<s>"}:
            continue
        ng = context + (w,)
        p = prob_addk(ng, ngram_counts, context_counts, V, k=k_smooth)
        candidates.append((w, p))
    candidates.sort(key=lambda x: -x[1])
    return candidates[:top_k]

# Bigram: context is 1 token
next_word_topk(["<s>"], n=2, ngram_counts=bi_counts, context_counts=bi_ctx, vocab=vocab, top_k=8)


[('<UNK>', 0.8076923076923077),
 ('</s>', 0.038461538461538464),
 ('not', 0.038461538461538464),
 ('after', 0.038461538461538464),
 ('error', 0.038461538461538464)]

## 8) Simple generation (bigram or trigram)


Text generation is not the main goal in SLMs, but it helps you verify:
- boundary handling
- smoothing
- OOV decisions

We will sample tokens until we hit `</s>`.


In [38]:

def sample_next(context_tokens: List[str], n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, k_smooth: float = 0.5):
    V = len(vocab)
    context = tuple(context_tokens[-(n-1):]) if n > 1 else tuple()
    words = [w for w in vocab if w != "<s>"]
    probs = []
    for w in words:
        ng = context + (w,)
        probs.append(prob_addk(ng, ngram_counts, context_counts, V, k=k_smooth))
    # Normalize
    s = sum(probs)
    probs = [p/s for p in probs]
    return random.choices(words, weights=probs, k=1)[0]

def generate(n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, max_len: int = 20, k_smooth: float = 0.5):
    tokens = ["<s>"]*(n-1) if n > 1 else []
    out = []
    for _ in range(max_len):
        w = sample_next(tokens, n, ngram_counts, context_counts, vocab, k_smooth=k_smooth)
        if w == "</s>":
            break
        out.append(w)
        tokens.append(w)
    return " ".join(out)

for _ in range(5):
    print("BIGRAM:", generate(2, bi_counts, bi_ctx, vocab, max_len=18))


BIGRAM: 
BIGRAM: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> not <UNK> <UNK> <UNK> <UNK>
BIGRAM: <UNK> <UNK> <UNK> <UNK>
BIGRAM: <UNK>
BIGRAM: after


## 9) Model comparison: effect of n and smoothing


Try different `k` values. Notes:
- `k=1.0` is Laplace smoothing (often too strong)
- smaller `k` (like 0.1 to 0.5) is often better

In real corpora, trigrams often beat bigrams, but require more data.


In [39]:

for k in [1.0, 0.5, 0.1, 0.01]:
    pp_bi_k  = evaluate_perplexity(test_texts, n=2, ngram_counts=bi_counts,  context_counts=bi_ctx,  vocab=vocab, k=k)
    pp_tri_k = evaluate_perplexity(test_texts, n=3, ngram_counts=tri_counts, context_counts=tri_ctx, vocab=vocab, k=k)
    print(f"k={k:>4}:  bigram PP={pp_bi_k:,.2f}   trigram PP={pp_tri_k:,.2f}")


k= 1.0:  bigram PP=1.95   trigram PP=2.09
k= 0.5:  bigram PP=1.87   trigram PP=1.96
k= 0.1:  bigram PP=1.79   trigram PP=1.80
k=0.01:  bigram PP=1.76   trigram PP=1.75


## Exercises (do these during lab)
1) Add 20 more realistic domain sentences to the corpus and re-run training/evaluation.  
2) Change `min_count` (OOV threshold) and explain how perplexity changes.  
3) Implement **backoff**: if a trigram is unseen, fall back to bigram; if unseen, fall back to unigram.  
4) Create a function that returns **top-5 next words** given a phrase like: `"user cannot"`.


### 1: Add more sentences to the corpus

In [40]:
new_sentences = [
    "database connection failed after server restart",
    "customer data not syncing across all modules",
    "mobile app crashes when uploading large files",
    "payment gateway timed out on checkout page",
    "vpn client not connecting from remote location",
    "network drive access denied for new users",
    "printer offline status persists despite troubleshooting",
    "email attachments not loading in web interface",
    "jira integration failing with authentication error",
    "server CPU usage consistently high overnight",
    "user profile images not displaying correctly",
    "calendar invites not sending to external recipients",
    "backup jobs failing with insufficient disk space",
    "application logs not being rotated regularly",
    "single sign on failing for specific department",
    "virtual machine performance very slow after update",
    "website forms not submitting data to backend",
    "reporting dashboard not refreshing with latest figures",
    "storage quota exceeded for several user accounts",
    "firmware update caused device to become unresponsive"
]

corpus.extend(new_sentences)

print(f"New corpus size: {len(corpus)}")

New corpus size: 34


In [41]:
# Re-split the corpus after adding new sentences
random.seed(42) # Keep seed for reproducibility
random.shuffle(corpus)
split = int(0.75 * len(corpus))
train_texts = corpus[:split]
test_texts = corpus[split:]

print(f"New train_texts length: {len(train_texts)}")
print(f"New test_texts length: {len(test_texts)}")
print(f"First two train texts: {train_texts[:2]}")
print(f"First two test texts: {test_texts[:2]}")

New train_texts length: 25
New test_texts length: 9
First two train texts: ['battery drains fast after bios update power settings unchanged', 'network drive access denied for new users']
First two test texts: ['email attachments not loading in web interface', 'wifi drops in meeting rooms access point reboot helps']


### 2: Change min_count (OOV threshold) and explain how perplexity changes

**min_count = 2**

In [42]:
# Build vocab from training data
train_tokens_flat = []
for t in train_texts:
    train_tokens_flat.extend(tokenize(t))

freq = Counter(train_tokens_flat)

# Typical practical rule: map tokens with frequency <= 1 to <UNK> in small corpora
min_count = 2
vocab = {w for w, c in freq.items() if c >= min_count}
vocab |= {"<UNK>", "<s>", "</s>"}

def replace_oov(tokens: List[str], vocab: set) -> List[str]:
    return [tok if tok in vocab else "<UNK>" for tok in tokens]

# Show OOV effect
sample = tokenize(test_texts[0])
sample, replace_oov(sample, vocab)

(['email', 'attachments', 'not', 'loading', 'in', 'web', 'interface'],
 ['<UNK>', '<UNK>', 'not', '<UNK>', '<UNK>', '<UNK>', '<UNK>'])

In [43]:
def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def train_ngram_counts(texts: List[str], n: int, vocab: set) -> Tuple[Counter, Counter]:
    ngram_counts = Counter()
    context_counts = Counter()
    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        for ng in get_ngrams(toks, n):
            ngram_counts[ng] += 1
            context = ng[:-1]
            context_counts[context] += 1
    return ngram_counts, context_counts

uni_counts, uni_ctx = train_ngram_counts(train_texts, n=1, vocab=vocab)
bi_counts, bi_ctx   = train_ngram_counts(train_texts, n=2, vocab=vocab)
tri_counts, tri_ctx = train_ngram_counts(train_texts, n=3, vocab=vocab)

len(uni_counts), len(bi_counts), len(tri_counts)

(15, 37, 64)

In [44]:
def evaluate_perplexity(texts: List[str], n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, k: float = 0.5) -> float:
    V = len(vocab)
    log2_probs = []
    token_count = 0

    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        ngrams = get_ngrams(toks, n)
        for ng in ngrams:
            p = prob_addk(ng, ngram_counts, context_counts, V, k=k)
            log2_probs.append(math.log(p, 2))
            token_count += 1

    H = -sum(log2_probs) / token_count
    PP = 2 ** H
    return PP

pp_uni = evaluate_perplexity(test_texts, n=1, ngram_counts=uni_counts, context_counts=uni_ctx, vocab=vocab, k=0.5)
pp_bi  = evaluate_perplexity(test_texts, n=2, ngram_counts=bi_counts,  context_counts=bi_ctx,  vocab=vocab, k=0.5)
pp_tri = evaluate_perplexity(test_texts, n=3, ngram_counts=tri_counts, context_counts=tri_ctx, vocab=vocab, k=0.5)

pp_uni, pp_bi, pp_tri

(2.35426857538057, 2.4711568126622905, 2.6400327897472438)

**min_count = 3**

In [45]:
# Build vocab from training data
train_tokens_flat = []
for t in train_texts:
    train_tokens_flat.extend(tokenize(t))

freq = Counter(train_tokens_flat)

# Typical practical rule: map tokens with frequency <= 1 to <UNK> in small corpora
min_count = 3
vocab = {w for w, c in freq.items() if c >= min_count}
vocab |= {"<UNK>", "<s>", "</s>"}

def replace_oov(tokens: List[str], vocab: set) -> List[str]:
    return [tok if tok in vocab else "<UNK>" for tok in tokens]

# Show OOV effect
sample = tokenize(test_texts[0])
sample, replace_oov(sample, vocab)

(['email', 'attachments', 'not', 'loading', 'in', 'web', 'interface'],
 ['<UNK>', '<UNK>', 'not', '<UNK>', '<UNK>', '<UNK>', '<UNK>'])

In [46]:
def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def train_ngram_counts(texts: List[str], n: int, vocab: set) -> Tuple[Counter, Counter]:
    ngram_counts = Counter()
    context_counts = Counter()
    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        for ng in get_ngrams(toks, n):
            ngram_counts[ng] += 1
            context = ng[:-1]
            context_counts[context] += 1
    return ngram_counts, context_counts

uni_counts, uni_ctx = train_ngram_counts(train_texts, n=1, vocab=vocab)
bi_counts, bi_ctx   = train_ngram_counts(train_texts, n=2, vocab=vocab)
tri_counts, tri_ctx = train_ngram_counts(train_texts, n=3, vocab=vocab)

len(uni_counts), len(bi_counts), len(tri_counts)

(10, 26, 44)

In [47]:
def evaluate_perplexity(texts: List[str], n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, k: float = 0.5) -> float:
    V = len(vocab)
    log2_probs = []
    token_count = 0

    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        ngrams = get_ngrams(toks, n)
        for ng in ngrams:
            p = prob_addk(ng, ngram_counts, context_counts, V, k=k)
            log2_probs.append(math.log(p, 2))
            token_count += 1

    H = -sum(log2_probs) / token_count
    PP = 2 ** H
    return PP

pp_uni = evaluate_perplexity(test_texts, n=1, ngram_counts=uni_counts, context_counts=uni_ctx, vocab=vocab, k=0.5)
pp_bi  = evaluate_perplexity(test_texts, n=2, ngram_counts=bi_counts,  context_counts=bi_ctx,  vocab=vocab, k=0.5)
pp_tri = evaluate_perplexity(test_texts, n=3, ngram_counts=tri_counts, context_counts=tri_ctx, vocab=vocab, k=0.5)

pp_uni, pp_bi, pp_tri

(2.1881907233817888, 2.2189449089944975, 2.297130431855181)

**min_count = 1**

In [48]:
# Build vocab from training data
train_tokens_flat = []
for t in train_texts:
    train_tokens_flat.extend(tokenize(t))

freq = Counter(train_tokens_flat)

# Typical practical rule: map tokens with frequency <= 1 to <UNK> in small corpora
min_count = 1
vocab = {w for w, c in freq.items() if c >= min_count}
vocab |= {"<UNK>", "<s>", "</s>"}

def replace_oov(tokens: List[str], vocab: set) -> List[str]:
    return [tok if tok in vocab else "<UNK>" for tok in tokens]

# Show OOV effect
sample = tokenize(test_texts[0])
sample, replace_oov(sample, vocab)

(['email', 'attachments', 'not', 'loading', 'in', 'web', 'interface'],
 ['email', '<UNK>', 'not', '<UNK>', 'in', '<UNK>', '<UNK>'])

In [49]:
def get_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def train_ngram_counts(texts: List[str], n: int, vocab: set) -> Tuple[Counter, Counter]:
    ngram_counts = Counter()
    context_counts = Counter()
    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        for ng in get_ngrams(toks, n):
            ngram_counts[ng] += 1
            context = ng[:-1]
            context_counts[context] += 1
    return ngram_counts, context_counts

uni_counts, uni_ctx = train_ngram_counts(train_texts, n=1, vocab=vocab)
bi_counts, bi_ctx   = train_ngram_counts(train_texts, n=2, vocab=vocab)
tri_counts, tri_ctx = train_ngram_counts(train_texts, n=3, vocab=vocab)

len(uni_counts), len(bi_counts), len(tri_counts)

(142, 190, 193)

In [50]:
def evaluate_perplexity(texts: List[str], n: int, ngram_counts: Counter, context_counts: Counter, vocab: set, k: float = 0.5) -> float:
    V = len(vocab)
    log2_probs = []
    token_count = 0

    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        toks = add_boundaries(toks, n)
        ngrams = get_ngrams(toks, n)
        for ng in ngrams:
            p = prob_addk(ng, ngram_counts, context_counts, V, k=k)
            log2_probs.append(math.log(p, 2))
            token_count += 1

    H = -sum(log2_probs) / token_count
    PP = 2 ** H
    return PP

pp_uni = evaluate_perplexity(test_texts, n=1, ngram_counts=uni_counts, context_counts=uni_ctx, vocab=vocab, k=0.5)
pp_bi  = evaluate_perplexity(test_texts, n=2, ngram_counts=bi_counts,  context_counts=bi_ctx,  vocab=vocab, k=0.5)
pp_tri = evaluate_perplexity(test_texts, n=3, ngram_counts=tri_counts, context_counts=tri_ctx, vocab=vocab, k=0.5)

pp_uni, pp_bi, pp_tri

(226.25848539874372, 148.32299753543973, 147.1426144707127)

In [51]:
oov_count_min1 = 0
all_test_tokens = []

# Recreate vocab with min_count = 1 to be safe, though it should already be set from previous runs
train_tokens_flat = []
for t in train_texts:
    train_tokens_flat.extend(tokenize(t))
freq = Counter(train_tokens_flat)
min_count = 1
vocab_min1 = {w for w, c in freq.items() if c >= min_count}
vocab_min1 |= {"<UNK>", "<s>", "</s>"}

for text in test_texts:
    toks = tokenize(text)
    oov_replaced_toks = replace_oov(toks, vocab_min1)
    all_test_tokens.extend(oov_replaced_toks)

oov_count_min1 = all_test_tokens.count("<UNK>")

print(f"Total OOV words (<UNK> tokens) in test_texts with min_count=1: {oov_count_min1}")

Total OOV words (<UNK> tokens) in test_texts with min_count=1: 46


Perplexity reduces slightly when min_count goes from 2 (2.354, 2.471, 2.640) to 3 (2.188, 2.219, 2.297), with the lowest being for unigram for both min_count. However, for min_count = 1, perplexity blows up to (226.259, 148.323, 147.143) and the lowest being for trigram, in this case.

This is because min_count = 3 has a smaller, but more robust vocabulary that reduces data sparsity for the frequent words, leading to better generalization. min_count = 2 strikes a balance by filtering out very rare words, but keeps more than min_count = 2. With min_count = 1, all the words that appear at least once make it into the vocabulary, causing it to be very large and sparse. This makes the model struggle to make meaningful predictions and gives a very high perplexity score.

### 3: Implement backoff: if a trigram is unseen, fall back to bigram; if unseen, fall back to unigram.

We'll define `get_backoff_prob` to recursively find the probability of an n-gram by falling back to `(n-1)`-grams if the current `n`-gram is unseen. This process continues until the unigram probability is reached. The `evaluate_perplexity_backoff` function will then use this new probability calculation.

In [52]:
def get_backoff_prob(
    ngram: Tuple[str, ...],
    uni_counts: Counter,
    uni_ctx: Counter,
    bi_counts: Counter,
    bi_ctx: Counter,
    tri_counts: Counter,
    tri_ctx: Counter,
    vocab: set,
    k: float = 0.5
) -> float:
    V = len(vocab)
    current_n = len(ngram)

    # Base case: Unigram
    if current_n == 1:
        return prob_addk(ngram, uni_counts, uni_ctx, V, k)

    # Try current n-gram order
    if current_n == 3:
        if tri_counts[ngram] > 0:
            return prob_addk(ngram, tri_counts, tri_ctx, V, k)
        else:
            # Fallback to (n-1)-gram: P(last_word | second_to_last_word)
            # Recursively call with the last (n-1) words of the ngram
            return get_backoff_prob(ngram[1:], uni_counts, uni_ctx, bi_counts, bi_ctx, tri_counts, tri_ctx, vocab, k)

    elif current_n == 2:
        if bi_counts[ngram] > 0:
            return prob_addk(ngram, bi_counts, bi_ctx, V, k)
        else:
            # Fallback to (n-1)-gram (unigram): P(last_word)
            # Recursively call with the last (n-1) words of the ngram
            return get_backoff_prob(ngram[1:], uni_counts, uni_ctx, bi_counts, bi_ctx, tri_counts, tri_ctx, vocab, k)

    else:
        raise ValueError(f"Ngram length {current_n} not supported for backoff function.")

In [53]:
def evaluate_perplexity_backoff(
    texts: List[str],
    n_model: int, # The highest order n-gram model to evaluate (e.g., 3 for trigram backoff)
    uni_counts: Counter, uni_ctx: Counter,
    bi_counts: Counter, bi_ctx: Counter,
    tri_counts: Counter, tri_ctx: Counter,
    vocab: set,
    k: float = 0.5
) -> float:
    log2_probs = []
    token_count = 0

    for text in texts:
        toks = replace_oov(tokenize(text), vocab)
        # Add boundaries according to the highest order model being evaluated (n_model)
        toks_with_boundaries = add_boundaries(toks, n_model)

        # Generate n-grams of the highest order from the text
        ngrams = get_ngrams(toks_with_boundaries, n_model)

        for ng in ngrams:
            # Calculate probability using the backoff strategy
            p = get_backoff_prob(ng, uni_counts, uni_ctx, bi_counts, bi_ctx, tri_counts, tri_ctx, vocab, k)

            if p <= 0: # Defensive: ensure probability is not zero for log
                p = 1e-10
            log2_probs.append(math.log(p, 2))
            token_count += 1

    H = -sum(log2_probs) / token_count
    PP = 2 ** H
    return PP

In [54]:
# Evaluate perplexity with backoff for bigram and trigram models

# First, ensure vocab, uni_counts, etc. are up-to-date with the desired min_count.
# Assuming we are using the 'min_count = 1' results from the previous run for comparison.
# You might want to re-run the vocab and counts with a different min_count if desired.

# For consistent comparison, let's explicitly re-run vocabulary and counts with min_count = 2 as initially suggested
# This is important as perplexity values change drastically with min_count

# Build vocab from training data with min_count = 2
train_tokens_flat = []
for t in train_texts:
    train_tokens_flat.extend(tokenize(t))

freq = Counter(train_tokens_flat)
min_count = 2
vocab_for_backoff = {w for w, c in freq.items() if c >= min_count}
vocab_for_backoff |= {"<UNK>", "<s>", "</s>"}

uni_counts_bo, uni_ctx_bo = train_ngram_counts(train_texts, n=1, vocab=vocab_for_backoff)
bi_counts_bo, bi_ctx_bo   = train_ngram_counts(train_texts, n=2, vocab=vocab_for_backoff)
tri_counts_bo, tri_ctx_bo = train_ngram_counts(train_texts, n=3, vocab=vocab_for_backoff)

# Evaluate with backoff
pp_bi_backoff = evaluate_perplexity_backoff(
    test_texts, n_model=2,
    uni_counts=uni_counts_bo, uni_ctx=uni_ctx_bo,
    bi_counts=bi_counts_bo, bi_ctx=bi_ctx_bo,
    tri_counts=tri_counts_bo, tri_ctx=tri_ctx_bo, # Trigram counts still passed for completeness, though not used in bigram backoff's first layer
    vocab=vocab_for_backoff, k=0.5
)

pp_tri_backoff = evaluate_perplexity_backoff(
    test_texts, n_model=3,
    uni_counts=uni_counts_bo, uni_ctx=uni_ctx_bo,
    bi_counts=bi_counts_bo, bi_ctx=bi_ctx_bo,
    tri_counts=tri_counts_bo, tri_ctx=tri_ctx_bo,
    vocab=vocab_for_backoff, k=0.5
)

print(f"Perplexity (Bigram with Backoff): {pp_bi_backoff:,.2f}")
print(f"Perplexity (Trigram with Backoff): {pp_tri_backoff:,.2f}")

# For comparison, let's also re-evaluate simple Add-k with the same min_count=2 vocab
pp_bi_addk = evaluate_perplexity(test_texts, n=2, ngram_counts=bi_counts_bo, context_counts=bi_ctx_bo, vocab=vocab_for_backoff, k=0.5)
pp_tri_addk = evaluate_perplexity(test_texts, n=3, ngram_counts=tri_counts_bo, context_counts=tri_ctx_bo, vocab=vocab_for_backoff, k=0.5)

print(f"Perplexity (Bigram Add-k): {pp_bi_addk:,.2f}")
print(f"Perplexity (Trigram Add-k): {pp_tri_addk:,.2f}")

Perplexity (Bigram with Backoff): 2.47
Perplexity (Trigram with Backoff): 2.59
Perplexity (Bigram Add-k): 2.47
Perplexity (Trigram Add-k): 2.64


#### Test `get_backoff_prob` with an example

Let's test the `get_backoff_prob` function with an example n-gram. We'll use a trigram that might not be directly observed in the training data to illustrate the fallback mechanism.

In [55]:
example_ngram_trigram = ("password", "reset", "<UNK>") # Example: 'password reset' followed by an OOV or rare word
example_ngram_bigram = ("reset", "<UNK>")
example_ngram_unigram = ("<UNK>",)

# Ensure '<UNK>' is in vocab, if not, it means min_count was too low or the word 'failed' was not OOV.
# Let's ensure a relevant vocab is used, for example, the vocab_for_backoff with min_count=2.

# Probability for the example trigram using backoff
prob_backoff_trigram = get_backoff_prob(
    example_ngram_trigram,
    uni_counts=uni_counts_bo, uni_ctx=uni_ctx_bo,
    bi_counts=bi_counts_bo, bi_ctx=bi_ctx_bo,
    tri_counts=tri_counts_bo, tri_ctx=tri_ctx_bo,
    vocab=vocab_for_backoff, k=0.5
)

# Direct probabilities for comparison (using prob_addk)
direct_prob_trigram = prob_addk(example_ngram_trigram, tri_counts_bo, tri_ctx_bo, len(vocab_for_backoff), k=0.5)
direct_prob_bigram = prob_addk(example_ngram_bigram, bi_counts_bo, bi_ctx_bo, len(vocab_for_backoff), k=0.5)
direct_prob_unigram = prob_addk(example_ngram_unigram, uni_counts_bo, uni_ctx_bo, len(vocab_for_backoff), k=0.5)

print(f"Example Trigram: {example_ngram_trigram}")
print(f"Backoff Probability for {example_ngram_trigram}: {prob_backoff_trigram:.4f}")
print(f"Direct Add-k Trigram Probability: {direct_prob_trigram:.4f}")
print(f"Direct Add-k Bigram Probability ({example_ngram_bigram}): {direct_prob_bigram:.4f}")
print(f"Direct Add-k Unigram Probability ({example_ngram_unigram}): {direct_prob_unigram:.4f}")

Example Trigram: ('password', 'reset', '<UNK>')
Backoff Probability for ('password', 'reset', '<UNK>'): 0.6330
Direct Add-k Trigram Probability: 0.0625
Direct Add-k Bigram Probability (('reset', '<UNK>')): 0.0625
Direct Add-k Unigram Probability (('<UNK>',)): 0.6330


The backoff model, when faced with unseen higher-order n-grams, gracefully falls back to a more general (lower-order) context that has more reliable statistics. This helps assign more realistic probabilities to unseen sequences compared to simply relying on the very small probabilities assigned by Add-k smoothing to a completely unseen n-gram, which can otherwise lead to an artificially inflated perplexity due to many near-zero probabilities.



### 4: Create a function that returns top-5 next words given a phrase like: "user cannot"

In [56]:
# Using the next_word_topk function defined in cell 11363d81

phrase = "user cannot"
context_tokens = tokenize(phrase)

# We'll use the bigram model (n=2) with the vocab and counts from the min_count=2 training.
# Ensure vocab_for_backoff, bi_counts_bo, bi_ctx_bo are still in scope or re-run relevant cells if needed.

top5_words = next_word_topk(
    context_tokens,
    n=2,
    ngram_counts=bi_counts_bo, # Using bigram counts from min_count=2
    context_counts=bi_ctx_bo,   # Using bigram context counts from min_count=2
    vocab=vocab_for_backoff,    # Using vocab from min_count=2
    k_smooth=0.5,
    top_k=5
)

print(f"Top 5 next words for '{phrase}':")
for word, prob in top5_words:
    print(f"- {word}: {prob:.4f}")


Top 5 next words for 'user cannot':
- </s>: 0.0625
- vpn: 0.0625
- for: 0.0625
- printer: 0.0625
- <UNK>: 0.0625


This output demonstrates how the smoothed n-gram model makes predictions, especially how it handles unseen or rare sequences by falling back to the smoothing constant, resulting in shared probabilities when actual counts are zero